In [ ]:
!pip install -q tensorflow opencv-python librosa matplotlib tqdm huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Google Drive folders
IMAGE_SOURCE = '/content/drive/MyDrive/deepfake_project/images'
VIDEO_SOURCE = '/content/drive/MyDrive/deepfake_project/videos'
AUDIO_SOURCE = '/content/drive/MyDrive/deepfake_project/audios'

# Where the .h5 models are stored on Drive
MODEL_DIR = '/content/drive/MyDrive/deepfake_models'
os.makedirs(MODEL_DIR, exist_ok=True)

print("Image source exists:", os.path.exists(IMAGE_SOURCE))
print("Video source exists:", os.path.exists(VIDEO_SOURCE))
print("Audio source exists:", os.path.exists(AUDIO_SOURCE))
print("Model dir:", MODEL_DIR)

Image source exists: True
Video source exists: True
Audio source exists: True
Model dir: /content/drive/MyDrive/deepfake_models


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications
import numpy as np
import cv2
import librosa
import matplotlib.pyplot as plt
import glob
import random
import shutil
from tqdm import tqdm
from sklearn.model_selection import train_test_split

IMG_SIZE = (299, 299)
BATCH_SIZE = 32
print("TensorFlow:", tf.__version__)
print("Libraries loaded.")

TensorFlow: 2.20.0
Libraries loaded.


In [ ]:
def create_image_dataset(data_dir):
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='binary'
    )
    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='binary'
    )
    return train_ds, val_ds


def build_image_model():
    base = applications.Xception(weights='imagenet', include_top=False, input_shape=(299,299,3))
    base.trainable = False
    inputs = layers.Input(shape=(299,299,3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-4),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

print("Image helpers ready.")

Image helpers ready.


In [ ]:
def extract_frames(video_path, num_frames=30):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        return []
    indices = np.linspace(0, total_frames-1, num_frames, dtype=int)
    frames = []
    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, (299, 299))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
    cap.release()
    return np.array(frames)


def create_video_list_direct(video_dir):
    data = []
    fake_dir = os.path.join(video_dir, 'deepfake')
    real_dir = os.path.join(video_dir, 'video')
    if os.path.isdir(fake_dir):
        for fname in os.listdir(fake_dir):
            full = os.path.join(fake_dir, fname)
            if os.path.isfile(full):
                data.append((full, 1))
    if os.path.isdir(real_dir):
        for fname in os.listdir(real_dir):
            full = os.path.join(real_dir, fname)
            if os.path.isfile(full):
                data.append((full, 0))
    return data

print("Video helpers ready.")

Video helpers ready.


In [ ]:
def load_audio(file_path, target_sr=16000, duration=4):
    signal, sr = librosa.load(file_path, sr=target_sr, duration=duration)
    if len(signal) < target_sr * duration:
        signal = np.pad(signal, (0, target_sr*duration - len(signal)))
    else:
        signal = signal[:target_sr*duration]
    return signal


def audio_to_melspectrogram(signal, sr=16000, n_mels=128, fmax=8000):
    mel = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=n_mels, fmax=fmax)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - np.min(log_mel)) / (np.max(log_mel) - np.min(log_mel) + 1e-8)
    return log_mel


def create_audio_list_direct(audio_dir):
    data = []
    real_dir = os.path.join(audio_dir, 'Original')
    fake_dir = os.path.join(audio_dir, 'ElevenLabs')
    if os.path.isdir(real_dir):
        for fname in os.listdir(real_dir):
            full = os.path.join(real_dir, fname)
            if os.path.isfile(full):
                data.append((full, 0))
    if os.path.isdir(fake_dir):
        for fname in os.listdir(fake_dir):
            full = os.path.join(fake_dir, fname)
            if os.path.isfile(full):
                data.append((full, 1))
    return data


def build_audio_model(input_shape):
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128, (3,3), activation='relu'),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

print("Audio helpers ready.")

Audio helpers ready.


In [ ]:
image_model_path = os.path.join(MODEL_DIR, 'image_deepfake_model.h5')

if os.path.exists(image_model_path):
    print("Loading existing image model...")
    image_model = keras.models.load_model(image_model_path, compile=False)
else:
    print("Training image model...")
    # Prepare small subset for speed
    small_dir = '/content/image_dataset_small'
    if not os.path.exists(small_dir):
        os.makedirs(f'{small_dir}/real', exist_ok=True)
        os.makedirs(f'{small_dir}/fake', exist_ok=True)
        for cls in ['real', 'fake']:
            src = os.path.join(IMAGE_SOURCE, cls)
            if os.path.isdir(src):
                files = os.listdir(src)
                sample = random.sample(files, min(500, len(files)))
                for f in sample:
                    shutil.copy2(os.path.join(src, f), f'{small_dir}/{cls}/{f}')

    train_ds, val_ds = create_image_dataset(small_dir)
    image_model = build_image_model()
    image_model.fit(train_ds, validation_data=val_ds, epochs=5)
    image_model.save(image_model_path)
    print(f"Image model saved to {image_model_path}")

Loading existing image model...


In [ ]:
video_model_path = os.path.join(MODEL_DIR, 'video_deepfake_model.h5')
video_data = create_video_list_direct(VIDEO_SOURCE)
print(f"Found {len(video_data)} videos.")

if os.path.exists(video_model_path):
    print("Loading existing video model...")
    video_model = keras.models.load_model(video_model_path, compile=False)
else:
    if len(video_data) > 0:
        base = applications.Xception(weights='imagenet', include_top=False, input_shape=(299,299,3))
        base.trainable = False
        x = base.output
        x = layers.GlobalAveragePooling2D()(x)
        feature_extractor = models.Model(inputs=base.input, outputs=x)

        def extract_video_features(video_path, num_frames=10):
            frames = extract_frames(video_path, num_frames)
            if len(frames) == 0:
                return None
            feats = []
            for f in frames:
                f = tf.keras.applications.xception.preprocess_input(f)
                fv = feature_extractor.predict(np.expand_dims(f, axis=0), verbose=0)
                feats.append(fv.flatten())
            return np.array(feats)

        X, y = [], []
        for path, label in tqdm(video_data, desc="Extracting features"):
            feats = extract_video_features(path, 10)
            if feats is not None:
                X.append(feats)
                y.append(label)

        X = np.array(X); y = np.array(y)
        if len(X) > 0:
            inp = layers.Input(shape=(10, 2048))
            x = layers.LSTM(64)(inp)
            x = layers.Dense(1, activation='sigmoid')(x)
            video_model = models.Model(inp, x)
            video_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
            video_model.fit(X, y, epochs=3, validation_split=0.2)
            video_model.save(video_model_path)
            print(f"Video model saved to {video_model_path}")
        else:
            print("No video features extracted.")
    else:
        print("No video data found.")

Found 11 videos.
Loading existing video model...


In [ ]:
audio_model_path = os.path.join(MODEL_DIR, 'audio_deepfake_model.h5')
audio_data = create_audio_list_direct(AUDIO_SOURCE)
print(f"Found {len(audio_data)} audio files.")

if os.path.exists(audio_model_path):
    print("Loading existing audio model...")
    audio_model = keras.models.load_model(audio_model_path, compile=False)
else:
    if len(audio_data) > 0:
        real = [(p, l) for p, l in audio_data if l == 0]
        fake = [(p, l) for p, l in audio_data if l == 1]
        subset = random.sample(real, min(200, len(real))) + random.sample(fake, min(200, len(fake)))
        random.shuffle(subset)

        X, y = [], []
        for path, label in tqdm(subset, desc="Processing audio"):
            try:
                sig = load_audio(path)
                mel = audio_to_melspectrogram(sig)
                mel = np.expand_dims(mel, axis=-1)
                X.append(mel); y.append(label)
            except Exception as e:
                print(f"Error: {e}")

        X = np.array(X); y = np.array(y)
        if len(X) > 0:
            audio_model = build_audio_model(input_shape=X.shape[1:])
            audio_model.fit(X, y, epochs=3, validation_split=0.2)
            audio_model.save(audio_model_path)
            print(f"Audio model saved to {audio_model_path}")
        else:
            print("No audio data loaded.")
    else:
        print("No audio data found.")

Found 1344 audio files.
Loading existing audio model...


In [ ]:
for name in ['image_deepfake_model.h5', 'video_deepfake_model.h5', 'audio_deepfake_model.h5']:
    path = os.path.join(MODEL_DIR, name)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ {name}  ({size_mb:.1f} MB)")
    else:
        print(f"❌ {name}  — MISSING")

✅ image_deepfake_model.h5  (79.9 MB)
✅ video_deepfake_model.h5  (6.2 MB)
✅ audio_deepfake_model.h5  (1.1 MB)


In [ ]:
from huggingface_hub import HfApi, login
import getpass

# Token from https://huggingface.co/settings/tokens (write permission)
HF_TOKEN = getpass.getpass("Paste your Hugging Face WRITE token: ")
login(token=HF_TOKEN)

api = HfApi()
repo_id = "Shivmehta04/deepfake-models"   # change username if needed

api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
print(f"Repo ready: https://huggingface.co/{repo_id}")

for name in ['image_deepfake_model.h5', 'video_deepfake_model.h5', 'audio_deepfake_model.h5']:
    path = os.path.join(MODEL_DIR, name)
    api.upload_file(
        path_or_fileobj=path,
        path_in_repo=name,
        repo_id=repo_id,
        repo_type="model"
    )
    print(f"✅ Uploaded {name}")

print("\nDirect download URLs:")
print(f"https://huggingface.co/{repo_id}/resolve/main/image_deepfake_model.h5")
print(f"https://huggingface.co/{repo_id}/resolve/main/video_deepfake_model.h5")
print(f"https://huggingface.co/{repo_id}/resolve/main/audio_deepfake_model.h5")

Paste your Hugging Face WRITE token: ··········
Repo ready: https://huggingface.co/Shivmehta04/deepfake-models


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...s/image_deepfake_model.h5:  25%|##4       | 20.8MB / 83.8MB            

✅ Uploaded image_deepfake_model.h5


...

✅ Uploaded video_deepfake_model.h5


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...s/audio_deepfake_model.h5:  49%|####9     |  568kB / 1.15MB            

✅ Uploaded audio_deepfake_model.h5

Direct download URLs:
https://huggingface.co/Shivmehta04/deepfake-models/resolve/main/image_deepfake_model.h5
https://huggingface.co/Shivmehta04/deepfake-models/resolve/main/video_deepfake_model.h5
https://huggingface.co/Shivmehta04/deepfake-models/resolve/main/audio_deepfake_model.h5


In [ ]:
import numpy as np

# Load each model
img_m = keras.models.load_model(os.path.join(MODEL_DIR, 'image_deepfake_model.h5'), compile=False)
print("Image model loaded. Input shape:", img_m.input_shape)

# Dummy inference
dummy = np.zeros((1, 299, 299, 3), dtype=np.float32)
pred = img_m.predict(dummy, verbose=0)
print("Dummy image prediction:", pred)

Image model loaded. Input shape: (None, 299, 299, 3)
Dummy image prediction: [[0.4509687]]
